## Step 3: Customer Segmentation (RFM)

### Goal
Segment customers based on purchasing behavior using Recency, Frequency, and Monetary metrics.

### Why this matters
RFM allows us to:
- Identify high-value (VIP) customers
- Detect churn risk
- Target marketing strategies effectively

In [1]:
import pandas as pd
from datetime import datetime

# Load cleaned dataset
df = pd.read_csv('../data/processed/ecommerce_cleaned.csv')

# Ensure datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Reference data (last date in dataset)
snapshot_date = df['InvoiceDate'].max()

# Create RFM table
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',  # Frequency
    'SalesAmount': 'sum'  # Monetary
}).reset_index()

# Rename columns
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,325,1,77183.60
1,12347.0,1,7,4310.00
2,12348.0,74,4,1797.24
3,12349.0,18,1,1757.55
4,12350.0,309,1,334.40


## RFM Scoring

### Goal
Convert raw Recency, Frequency, and Monetary values into standardized scores (1–5) to rank customers.

### Why this matters
- Allows comparison across customers
- Enables segmentation into meaningful groups (VIP, At Risk, etc.)
- Simplifies complex behavior into actionable insights

### Scoring Logic
- Recency: Lower is better → reversed scoring (recent customers get higher scores)
- Frequency: Higher is better → frequent buyers get higher scores
- Monetary: Higher is better → high spenders get higher scores

In [2]:
# Create RFM scores (1-5)

rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]) # lower recency = better -> higher score
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]) # higher frequency = better -> higher score
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]) # higher monetary = better -> higher score

# Combine into one RFM score
rfm['RFM_score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score
0,12346.0,325,1,77183.60,1,1,5,115
1,12347.0,1,7,4310.00,5,5,5,555
2,12348.0,74,4,1797.24,2,4,4,244
3,12349.0,18,1,1757.55,4,1,4,414
4,12350.0,309,1,334.40,1,1,2,112


## Customer Segmentation

### Goal
Group customers into segments based on RFM scores.

### Why this matters
- Identifies high-value customers (VIPs)
- Detects customers at risk of churning
- Enables targeted marketing strategies

### Segment Definitions
- VIP: High recency, frequency, and monetary value (555)
- Loyal: Frequent and recent buyers
- At Risk: Previously frequent but not recent buyers
- Regular: All other customers

In [7]:
def segment_customer(row):
    if int(row["R_score"]) >= 4 and int(row["F_score"]) >= 4 and int(row["M_score"]) >= 4:
     return "VIP"
    elif row["R_score"] >= 4 and row["F_score"] >= 4:
        return "Loyal"
    elif row["R_score"] <= 2 and row["F_score"] >= 3:
        return "At Risk"
    else:
        return "Regular"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

rfm["Segment"].value_counts()

Segment
Regular    2556
VIP         962
At Risk     643
Loyal       177
Name: count, dtype: int64

## Revenue Contribution by Segment

### Goal
Understand which customer segments drive the most total revenue.

### Why this matters
- Identifies where the business makes money
- Helps prioritize high-value customer groups
- Supports strategic decisions (retention vs acquisition)

### Key Insight
- VIP customers contribute the majority of total revenue
- Regular customers contribute significant volume but lower value per customer
- At Risk and Loyal segments contribute less overall revenue

### Business Implication
- Revenue is highly concentrated among VIP customers
- The business should prioritize retaining and protecting VIP customers

In [8]:
rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)

Segment
VIP        5809341.070
Regular    2183152.442
At Risk     800531.551
Loyal       118382.841
Name: Monetary, dtype: float64

## Average Customer Value by Segment

### Goal
Measure the average revenue generated per customer within each segment.

### Why this matters
- Reveals customer quality, not just volume
- Helps identify which segments are most valuable on a per-customer basis
- Supports targeting strategies for marketing and retention

### Key Insight
- VIP customers have the highest average spend (~$6,000+ per customer)
- At Risk customers still show relatively high value, indicating recoverable revenue
- Regular and Loyal customers have lower average spend

### Business Implication
- VIP customers are both high-value and high-impact
- At Risk customers represent an opportunity for revenue recovery
- Regular customers require nurturing to increase value over time

In [9]:
rfm.groupby("Segment")["Monetary"].mean().sort_values(ascending=False)

Segment
VIP        6038.816081
At Risk    1244.994636
Regular     854.128498
Loyal       668.829610
Name: Monetary, dtype: float64

## Overall Segment Strategy

### Summary
- VIP customers drive the majority of revenue and have the highest value
- Regular customers represent the largest group but lowest engagement
- At Risk customers present a key opportunity for revenue recovery

### Strategic Focus
1. Protect VIP customers (loyalty programs, exclusive offers)
2. Re-engage At Risk customers (targeted campaigns)
3. Develop Regular customers into repeat buyers